ST554 Project 2

Franklin Zhou

Mar 9 2026

# Part I


## Introduction

I'm going to use the `SparkDataCheck.py` to do the following tasks:

- validation checks

- summarization functions

- chainable operations

## Initiating 

First, import the script, required modules and initiate the spark session:

In [1]:
from SparkDataCheck import SparkDataCheck
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("AirDatasetCheck") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 20:25:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Prepare for data

Read in the air quality data and make some data cleaning and transformation. Since the table is too wide, I select some variables and export to a `.csv` file as an example.

In [2]:
air_data = pd.read_csv("air.csv")

# change all "-200" to missing
air_data.replace(-200, np.nan, inplace = True)

# create some categorical variables
air_data["Time_TS"] = air_data.Time.astype('timedelta64[ns]')
air_data["Time_of_Day"] = pd.cut(air_data.Time_TS, 3, labels = ["Early", "Mid", "Late"])
air_data["Temp_Range"] = pd.cut(air_data['T'], bins = [-20,0,15,30,50], labels = ["Cold", "Cool", "Warm", "Hot"])

# Pick a subset of the data as an illustration and export to .csv file
air_data.loc[:, ["CO(GT)", "NO2(GT)", "C6H6(GT)", "Time_of_Day", "Temp_Range"]].to_csv('air_data_cleaned.csv', index = False)

In [ ]:
Read the `.csv` file by using `.read_csv()` method

In [3]:
air = SparkDataCheck.read_csv(spark, "air_data_cleaned.csv")

## Examine Validation methods

Then I'll check all methods that I created in the class.

### 1. Validate numeric bounds

In [4]:
# Is the value of C6H6(GT) is between 5 and 9
air.check_numeric("NO2(GT)", lower = 50, upper = 100)
air.df.show()

+------+-------+--------+-----------+----------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|
+------+-------+--------+-----------+----------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                false|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|
|   2.2|  114.0|     9.0|       Late|      Cool|                false|
|   2.2|  122.0|     9.2|       Late|      Cool|                false|
|   1.6|  116.0|     6.5|       Late|      Cool|                false|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|
|   1.0|   76.0|     3.3|      Early|      Cool|                 true|
|   0.9|   60.0|     2.3|      Early|      Cool|                 true|
|   0.6|   NULL|     1.7|      Early|      Cool|                 NULL|
|  NULL|   34.0|     1.3|      Early|      Cool|                false|
|   0.

In [32]:
# Is the value of C6H6(GT) is above 50
air.check_numeric("NO2(GT)", lower = 50)
air.df.show()

+------+-------+--------+-----------+----------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|
+------+-------+--------+-----------+----------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|
|   1.0|   76.0|     3.3|      Early|      Cool|                 true|
|   0.9|   60.0|     2.3|      Early|      Cool|                 true|
|   0.6|   NULL|     1.7|      Early|      Cool|                 NULL|
|  NULL|   34.0|     1.3|      Early|      Cool|                false|
|   0.

In [33]:
# If the input column is not numeric
air.check_numeric("Temp_Range", lower = 50)

### 2. Validate string type

In [34]:
air.check_string("Time_of_Day", levels = ["Mid","Late"])
air.df.show()

+------+-------+--------+-----------+----------+---------------------+------------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|Time_of_Day_string_check|
+------+-------+--------+-----------+----------+---------------------+------------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|                    true|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|                    true|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|                    true|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|                    true|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|                    true|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|                    true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|                   false|
|   1.0|   76.0|     3.3|      Early|   

In [35]:
air.check_string("NO2(GT)")

### 3. Validate missing values

In [36]:
air.check_missing("NO2(GT)")
air.df.show()

+------+-------+--------+-----------+----------+---------------------+------------------------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|Time_of_Day_string_check|NO2(GT)_missing_check|
+------+-------+--------+-----------+----------+---------------------+------------------------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|                    true|                false|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|                    true|                false|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|                    true|                false|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|                    true|                false|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|                    true|                false|
|   1.2|   96.0|     4.7|       Late|      Cool|        

## Examine Summarization methods

### 1. find min and max of column(s)

In [5]:
# find min and max of NO2(GT)
air.min_max("NO2(GT)")

,min(NO2(GT)),max(NO2(GT))
0,2.0,340.0


In [6]:
# find min and max of NO2(GT) grouped by temp range
air.min_max("NO2(GT)", group = "Temp_Range")

,Temp_Range,min(NO2(GT)),max(NO2(GT))
0,Cool,13.0,333.0
1,Cold,27.0,169.0
2,None,35.0,340.0
3,Hot,14.0,233.0
4,Warm,2.0,272.0


In [8]:
# find min and max of all numeric columns gruped by Time of Day
air.min_max(group = "Time_of_Day")

,Time_of_Day,min(CO(GT)),max(CO(GT)),min(NO2(GT)),max(NO2(GT)),min(C6H6(GT)),max(C6H6(GT))
0,Late,0.1,11.9,21.0,340.0,1.3,52.1
1,Early,0.1,5.9,2.0,250.0,0.1,31.5
2,Mid,0.1,8.1,5.0,333.0,0.3,63.7


In [9]:
# if column is not numeric type, display warning message
air.min_max("Time_of_Day")

### 2. count by column levels

In [10]:
# by one column
air.levels_count("Time_of_Day")

,Time_of_Day,count
0,Late,3118
1,Early,3120
2,Mid,3119


In [11]:
# by two columns
air.levels_count("Time_of_Day", "Temp_Range")

,Time_of_Day,Temp_Range,count
0,Early,Warm,1465
1,Mid,Warm,1493
2,Mid,Hot,537
3,Early,Cold,11
4,Late,Cool,1018
5,Mid,Cool,976
6,Late,None,139
7,Early,None,117
8,Early,Cool,1527
9,Late,Hot,401


In [12]:
# if any column is numeric
air.levels_count("Time_of_Day", "NO2(GT)")

## Using pandas data frame

In [26]:
# Pick a subset of the data as an illustration and export to .csv file
air_data_pd = air_data.loc[:, ["CO(GT)", "NO2(GT)", "C6H6(GT)", "Time_of_Day", "Temp_Range"]]
isinstance(air_data_pd, pd.DataFrame) # varified that this is a pandas dataframe

True

In [16]:
# create an instance from pandas dataframe
air_pd =  SparkDataCheck.from_pandas_df(spark, air_data_pd)

In [27]:
# call the count method
air_pd.levels_count("Time_of_Day", "Temp_Range")

,Time_of_Day,Temp_Range,count
0,Mid,Warm,1493
1,Late,Cool,1018
2,Mid,Cool,976
3,Early,Cool,1527
4,Late,Warm,1560
5,Early,Warm,1465
6,Late,NaN,139
7,Mid,NaN,110
8,Early,NaN,117
9,Mid,Hot,537


In [ ]:
# Part II